Wikipedia Retriever : It feels like it is a document loader but it is retriever because it brings data (via wikipedia api) using some sort of searching , and brings only relevant info instead of bringing all the documents, since it has some type of intelligence , it is a retriver and not a document loader

In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [2]:
retriever=WikipediaRetriever(
    top_k_results=2,
    lang='en'
)

In [3]:
query='the impact of ai on technology'

In [4]:
docs=retriever.invoke(query)
# retriver object is a runnable as it has invoke function

In [7]:
for i , doc in enumerate(docs):
    print(f'result -> {i+1} \n')
    print(f'content-> {doc.page_content}')

result -> 1 

content-> The India AI Impact Summit 2026 is an event for artificial intelligence from 16 to 21 February 2026, sponsored by the Indian government and held at Bharat Mandapam in New Delhi, India. It is the fourth in a series of global AI summits following the AI Safety Summit in Bletchley Park in 2023, the AI Seoul Summit in 2024, the AI Action Summit in Paris in 2025, and the succeeding AI Impact Summit in Geneva in 2027. Organised under the IndiaAI Mission by the Ministry of Electronics and Information Technology, it is the first summit in the series to be hosted by a Global South nation.
The summit was inaugurated by Prime Minister Narendra Modi on 19 February 2026. The opening ceremony was also addressed by French President Emmanuel Macron and United Nations Secretary-General António Guterres.
The summit drew delegations from over 100 countries, including more than 20 heads of states, 60 ministers, and nearly 300,000 participants. It featured more than 300 curated exhi

Vector Store Retriever: Search and fetch documents from vector store based on semantic similarity using vector embeddings

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [9]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [10]:
embedding_model=HuggingFaceEmbeddings()

vector_store=Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name='my_collection'
)

In [42]:
retriever1=vector_store.as_retriever(search_kwargs={"k":2})

In [43]:
query="what is chroma used for?"
result1=retriever1.invoke(query)

In [44]:
for i , doc in enumerate(result1):
    print(f'result {i+1}')
    print(f'content: {doc.page_content}')

result 1
content: Chroma is used to store and search document embeddings.
result 2
content: LangChain supports Chroma, FAISS, Pinecone, and more.


In [35]:
result2=vector_store.similarity_search(query)
for i , doc in enumerate(result2):
    print(f'result {i+1}')
    print(f'content: {doc.page_content}')

result 1
content: Chroma is a vector database optimized for LLM-based search.
result 2
content: Embeddings convert text into high-dimensional vectors.
result 3
content: LangChain helps developers build LLM applications easily.
result 4
content: OpenAI provides powerful embedding models.


vector store also has the capanility to do similarity search but it has only one criteria or strategy of searching , but if we want to use different strategy , then we need retrievers.
we can create advance type of retrievers which uses advance strategies of searching and are also runnables therefore can be connected to chains

Maximal Marginal Relevance (MMR):Picks document which are not only relevant to the query but are also different from the previously picked document i.e. no redundancy

In [4]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
from langchain_community.vectorstores import FAISS
embedding_model=HuggingFaceEmbeddings()
vector_store1=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [39]:
retriever_1 = vector_store.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  
)

when "lambda_mult":1 then mmr behaves like simple vector store retriever 

In [40]:
query='What is langchain?'
results_1=retriever_1.invoke(query)
print(query)


What is langchain?


In [41]:
for i, doc in enumerate(results_1):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
Embeddings are vector representations of text.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


Multi Query Retriever:USed when the query is ambiguious , when it is used , the query is send to llm which generates more and different queries which covers same point to remove ambiguity.Then these query are sent to normal retriever which search in a common vector store and merges result of all queries.

In [2]:
from langchain_classic.retrievers import MultiQueryRetriever

c:\Users\DELL\OneDrive\Desktop\LangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
from langchain_huggingface import ChatHuggingFace
from dotenv import load_dotenv

In [24]:
load_dotenv()

True

In [25]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [26]:
embedding_model=HuggingFaceEmbeddings()
vector_store=FAISS.from_documents(documents=all_docs,embedding=embedding_model)

In [27]:
# creating normal retriever
similarity_retriever=vector_store.as_retriever(search_type='similarity' , search_kwargs={"k":5})

In [28]:
from langchain_huggingface import HuggingFaceEndpoint

In [29]:
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task='text-generation'
)

model=ChatHuggingFace(llm=llm)


In [30]:
chat_model=model

In [31]:
multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k":5}),
    llm=chat_model
)

In [32]:
query = "How to improve energy levels and maintain balance?"

In [33]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [34]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
The solar energy system in modern homes helps balance electri

Contextual Compression Retriever:it improves retrieval quality by compressing documents after retrieval , keeping only the relevant content based on user's query

In [35]:
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [36]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [37]:
embedding_model=HuggingFaceEmbeddings()
vector_store=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [40]:
base_retriever=vector_store.as_retriever(search_kwargs={'k':5})

In [38]:
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task='text-generation'
)

llm=ChatHuggingFace(llm=llm)
compressor = LLMChainExtractor.from_llm(llm)

In [41]:
compression_retrieval=ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [43]:
query = "What is photosynthesis?"
compressed_results = compression_retrieval.invoke(query)

In [44]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Extracted relevant parts:
        Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
